<a href="https://colab.research.google.com/github/goutham3010/Hospital-Patient-Flow-and-Resource-Demand-Prediction-System/blob/main/08_Clinical_Decision_Support.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HOSPITAL PATIENT FLOW & DEMAND PREDICTION SYSTEM
## NOTEBOOK 08: RAG & CLINICAL DECISION SUPPORT
**Workflow Phases 13, 14 & 15** of the 17-Stage Hospital Operations Decision Support Architecture.

### Objectives:
1. Index Hospital Standard Operating Procedures (SOPs) into a searchable vector knowledge base.
2. Implement **Retrieval-Augmented Generation (RAG)** using TF-IDF cosine similarity.
3. Automatically determine **Capacity Surge Escalation Levels** (Green, Yellow, Amber, Red/Purple).
4. Compute **Nurse-to-Patient Staffing Quotas** across General, ICU, HDU, and NICU wards.
5. Generate an **Executive Clinical Operations Briefing** and tactical action checklist.


In [ ]:
# 1. IMPORT LIBRARIES & RAG ENGINE
import pandas as pd
import numpy as np
import sys
import os

sys.path.append('src')
from rag_engine import HospitalRAGEngine
from llm_assistant import ClinicalLLMAssistant

print("RAG and Decision Support engines loaded.")


In [ ]:
# 2. INITIALIZE AND QUERY RAG SOP KNOWLEDGE BASE
rag = HospitalRAGEngine('data/hospital_sops.json')
print(f"Indexed {len(rag.sops)} hospital Standard Operating Procedures.")

# Test query for capacity surge
query = "Hospital occupancy 94%, ICU beds saturated, emergency admissions high"
retrieved_sops = rag.retrieve(query, top_k=3)

print("=== RELEVANT HOSPITAL PROTOCOLS RETRIEVED ===")
for r in retrieved_sops:
    print(f" • [{r['id']}] {r['title']} (Score: {r['relevance_score']:.3f})")


In [ ]:
# 3. FORMAT RETRIEVED PROTOCOLS FOR DECISION SUPPORT
formatted_context = rag.format_sop_context_for_llm(retrieved_sops)
print(formatted_context[:600] + "...\n[Truncated]")


In [ ]:
# 4. EXECUTE CLINICAL DECISION SUPPORT & STAFFING DIRECTIVES
assistant = ClinicalLLMAssistant()

# Simulate high surge scenario
briefing = assistant.generate_operations_briefing(
    predicted_demand=44.0,
    baseline_demand=32.8,
    ward_predictions={'general': 28.0, 'icu': 8.0, 'hdu': 5.0, 'nicu': 3.0},
    shap_explanation={'positive_drivers': [{'label': '7-Day Rolling Surge', 'shap_value': 4.5}], 'negative_drivers': []},
    rag_sops=retrieved_sops,
    current_census=295,
    total_capacity=320
)

print(f"SURGE STATUS: {briefing['surge_code']}")
print(f"Projected Occupancy: {briefing['predicted_occupancy_rate']}%\n")

print("=== WARD STAFFING DIRECTIVES (SOP-STAFF-02) ===")
for ward, nurses in briefing['staffing_requirements'].items():
    print(f" • {ward.upper():<16}: {nurses} RNs")

print("\n=== IMMEDIATE INCIDENT COMMAND CHECKLIST ===")
for act in briefing['action_checklist']:
    print(f" • {act}")
